# Nemotron SFT Training (Kaggle / RTX PRO 6000)

Runs the convergent public 0.85 SFT recipe + `completion_only_loss=True`. Defaults to a **smoke run** (200 rows, 50 steps) — bump `sample_n` to `None` and `max_steps` to `-1` for a full ~1-epoch run on the kienngx CoT corpus.

Data sources required: this code dataset (`aleixlopez/nemotron-code`), the base model (`metric/nemotron-3-nano-30b-a3b-bf16/transformers/default`), and the kienngx CoT corpus (`kienngx/nemotron-30b-competition-trainingdata-cot-labels`).


In [ ]:
# Set memory env BEFORE torch import (cannot be changed after).
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Untar our code bundle into /kaggle/working so the package layout is preserved.
import subprocess, sys
subprocess.run(
    ["tar", "xzf", f"/kaggle/input/nemotron-code/nemotron-code.tar.gz", "-C", "/kaggle/working"],
    check=True,
)
sys.path.insert(0, "/kaggle/working")

# Sanity: our packages import.
import nemotron_local_cv  # noqa: F401
import nemotron_train  # noqa: F401
print("nemotron_local_cv + nemotron_train imported.")


In [ ]:
# Apply pre-model-load Blackwell env. (post-load patches happen after model load.)
from kaggle.blackwell_setup import set_cuda_alloc_env, set_blackwell_ptxas_env  # noqa: E402
set_cuda_alloc_env()
set_blackwell_ptxas_env()
print("CUDA alloc env + Blackwell ptxas env set.")


In [ ]:
# Training-only deps. Kaggle's base image ships torch + transformers + datasets;
# we add the exact trl/peft/accelerate versions that match our recipe.
import subprocess, sys
deps = [
    "trl>=0.12,<0.18",
    "peft>=0.13,<0.16",
    "accelerate>=1.0",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *deps])


In [ ]:
import kagglehub
from pathlib import Path

BASE_MODEL_PATH = Path(kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"))
print(f"BASE_MODEL_PATH: {BASE_MODEL_PATH}")

DATA_DIR = Path("/kaggle/working/data")
VAL_SPLIT = DATA_DIR / "val_split_600.csv"
assert VAL_SPLIT.exists(), VAL_SPLIT


In [ ]:
# Locate the kienngx CoT corpus dataset. Attach it to this kernel before running:
#   Dataset slug: kienngx/nemotron-30b-competition-trainingdata-cot-labels
from pathlib import Path
import glob

CORPUS_CANDIDATES = glob.glob("/kaggle/input/**/final_Nemotron_training_data.csv", recursive=True)
assert CORPUS_CANDIDATES, "Attach the kienngx CoT corpus dataset before running."
CORPUS_PATH = Path(CORPUS_CANDIDATES[0])
print(f"CORPUS_PATH: {CORPUS_PATH}")

OUTPUT_DIR = Path("/kaggle/working/adapter")
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
# Smoke run first; flip --sample-n / --max-steps for the full run.
from nemotron_train import train

train(
    data_csv=str(CORPUS_PATH),
    output_dir=str(OUTPUT_DIR),
    base_model=str(BASE_MODEL_PATH),
    sample_n=200,        # bump to None for full corpus
    max_steps=50,        # bump to -1 for epochs-driven full run (~1000 steps)
)


In [ ]:
# Package the adapter as submission.zip in /kaggle/working (the Kaggle "Submit" button
# zips notebook outputs; this matches the host's expected zip layout).
import zipfile, os
from pathlib import Path

ADAPTER_DIR = OUTPUT_DIR  # already a Path
ZIP_PATH = Path("/kaggle/working/submission.zip")

REQUIRED = ["adapter_config.json", "adapter_model.safetensors"]
present = set(os.listdir(ADAPTER_DIR))
missing = [f for f in REQUIRED if f not in present]
assert not missing, f"missing required adapter files: {missing}"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in present:
        zf.write(ADAPTER_DIR / f, arcname=f)
print(f"wrote {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1024 / 1024:.2f} MB)")
